In [167]:
from google.colab import files
uploaded = files.upload()  # CSV 파일 업로드 대화창이 뜹니다

Saving modeling_data_generated_large.csv to modeling_data_generated_large (1).csv


In [163]:
import pandas as pd
import numpy as np

# 1) 데이터 로드
df = pd.read_csv("modeling_data_generated_large.csv")

# 2) 점수 계산 (FanLevel 내 FalseAlarm 정규화)
EPS = 1e-9
ALPHA = 3.0
BETA  = 1.0

def add_score(_df):
    fa_min = _df.groupby("FanLevel")["FalseAlarm"].transform("min")
    fa_max = _df.groupby("FanLevel")["FalseAlarm"].transform("max")
    fa_norm = (_df["FalseAlarm"] - fa_min) / (fa_max - fa_min + EPS)
    out = _df.copy()
    out["FA_Norm"] = fa_norm.clip(0,1)
    out["Score"]   = ALPHA * out["TrueDetectionRate"] - BETA * out["FA_Norm"]
    return out

df_scored = add_score(df)

# 3) FanLevel별 최적 민감도 계산
def pick_best(g):
    return g.sort_values(
        ["Score","FA_Norm","TrueDetectionRate","SensorSensitivity"],
        ascending=[False, True, False, True]
    ).iloc[0]

best = (df_scored.groupby("FanLevel", as_index=False)
                  .apply(pick_best)
                  .reset_index(drop=True))

summary = best[["FanLevel","SensorSensitivity","TrueDetectionRate","FalseAlarm","Score"]].copy()
summary.rename(columns={"SensorSensitivity":"OptimalSensitivity"}, inplace=True)
summary["TrueDetectionRate"] = summary["TrueDetectionRate"].round(3)
summary["FalseAlarm"] = summary["FalseAlarm"].round(1)
summary["Score"] = summary["Score"].round(6)

# 4) 결과 출력
print(summary.to_string(index=False))

 FanLevel  OptimalSensitivity  TrueDetectionRate  FalseAlarm    Score
      0.0                 7.0              0.977        20.9 2.352233
      1.0                 5.0              0.979        37.7 2.448062
      2.0                 4.0              0.922        56.4 2.428885
      3.0                 4.0              0.918        84.8 2.372677


/tmp/ipython-input-3315303292.py:31: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(pick_best)


In [164]:
# 전체 데이터에 라벨 부착해 저장
best_map = dict(zip(best["FanLevel"], best["SensorSensitivity"]))
df_labeled = df_scored.copy()
df_labeled["OptimalSensitivity"] = df_labeled["FanLevel"].map(best_map).astype(int)
df_labeled.to_csv("model_labeled_dataset.csv", index=False)

In [166]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import joblib

# 1) 데이터 로드
df = pd.read_csv("model_labeled_dataset.csv")

# 2) Feature/Target
X = df[["FanLevel"]]                      # 현재 단계: FanLevel만 사용
y = df["OptimalSensitivity"].astype(int)

# 3) 학습/검증 분리
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# 4) RF 학습
clf = RandomForestClassifier(
    n_estimators=400,
    max_depth=None,
    min_samples_split=4,
    random_state=42,
    n_jobs=-1
)
clf.fit(X_train, y_train)

# 5) 평가
y_pred = clf.predict(X_test)
print("Accuracy:", round(accuracy_score(y_test, y_pred), 4))
print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred, zero_division=0))

# 6) 예측 예시
print(clf.predict(pd.DataFrame({"FanLevel":[0,1,2,3]})))

Accuracy: 1.0
[[180   0   0]
 [  0  90   0]
 [  0   0  90]]
              precision    recall  f1-score   support

           4       1.00      1.00      1.00       180
           5       1.00      1.00      1.00        90
           7       1.00      1.00      1.00        90

    accuracy                           1.00       360
   macro avg       1.00      1.00      1.00       360
weighted avg       1.00      1.00      1.00       360

[7 5 4 4]
